# 10. TOP3 매수 신호 CSV 점검 (팀원 매도 ML·DL 용)

대상: 팀원 4명 (매도 신호를 각자 ML·DL 로 만들 때 쓰는 매수 신호 파일이 맞게 만들어졌는지 확인)

- 11/26 회의 결정: 매수는 PDF 순위로 시장별 TOP3 (모두 H5), 매도는 팀원이 각자 ML·DL 로 확장 창 워크포워드
  (2011~2015 학습 → 2016 시험, … , 2011~2018 학습 → 2019 시험). 2020 이후는 쓰지 않는다.
- 규칙은 **실행 전에** `실험기록.csv` 36번째 줄에 적었다. 결과는 37번째 줄. 새로 고른 것은 없다 (시도 수 0).
- 명령: `python -m indexbeater top3csv --run` → `results/top3_buy_signals.csv` (+ `_check.csv`, `_07_final.csv`).
- 엔진·점수표 파일은 고치지 않고 부르기만 했다. 2020년 이후 자료는 열지 않았다.

1. 테스트
2. CSV 모양 (열·줄 수·기간)
3. 신호를 만든 방법 (해마다 / 최종 모델)
4. CSV 만 다시 읽어 엔진에 넣기 → PDF 포트폴리오와 같은가 (이 노트북에서 직접 다시 계산)
5. 다-2 최종 모델 진단
6. 팀원이 쓸 때 주의할 것
7. 정리: 확인한 것 / 확인 못한 것 / 이상한 것

이 노트북은 데이터 파일을 저장하지 않는다. 결과 파일은 명령만 만든다.

In [1]:
# 이 칸을 맨 먼저 실행한다.
# 노트북이 어느 폴더에서 열려도, indexbeater 폴더가 있는 프로젝트 폴더를 찾아서 길 목록(sys.path)에 넣는다.
import os
import sys

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "indexbeater")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)          # 못 찾으면 한 칸 위 폴더로 올라가서 다시 찾는다
sys.path.insert(0, root)
os.chdir(root)
print("프로젝트 폴더:", root)

프로젝트 폴더: C:\Users\gram\Desktop\project\buysignal


## 1. 테스트

`tests/check_period.py` 중 TOP3 CSV 테스트 4개만 돌린다 (기본 pytest 130개 모음에는 넣지 않음).
- `test_top3csv_final_cut`: 최종 모델을 고른 날 = 2020 첫날 − 21거래일 (점수표의 2020 cut 과 같음). 2020 자료가 있으면 멈춤
- `test_top3csv_final_07_window`: 다-2 최종 모델 학습 = cut 까지 신호 · 2020 첫날 전에 팔린 거래만. 점수 매긴 후보는 2015 전만
- `test_top3csv_splice`: 2012~2014 줄 = 최종 모델, 2015~ 줄 = 해마다 고른 것, 다른 시장·엔진이 안 보는 칸은 0
- `test_top3csv_rows_roundtrip`: 매수가 = 다음 날 수정 시가, 상태 = 엔진과 같음, CSV 로 저장했다 다시 읽어도 같은 종목을 삼

맨 아래 줄이 `passed` 이고 `failed` 가 없으면 통과.

In [2]:
import subprocess

r = subprocess.run([sys.executable, "-m", "pytest", "tests/check_period.py", "-k", "top3csv", "--color=no", "-q"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:] or r.stderr[-1500:])

....                                                                     [100%]



## 2. CSV 모양

한 줄 = 신호 하나 (시장·전략·신호일·종목). 20자리 제한 **전에** 켜진 신호 전부를 넣었다.
엔진이 처음부터 보지 않는 칸 (유동성 필터 통과 못 함·2012-07-01 전) 은 뺐다.

In [3]:
import numpy as np
import pandas as pd
from indexbeater import buy_export as BX
from indexbeater import config as C

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
df = pd.read_csv(BX.OUT, dtype={"종목코드": str})
print("줄 수:", f"{len(df):,}", "· 열 수:", df.shape[1])
print("열:", list(df.columns))
print("신호일:", df["신호일"].min(), "~", df["신호일"].max())
print("종목코드 6자리:", bool(df["종목코드"].str.len().eq(6).all()))
print()
print(df.groupby(["시장", "TOP3순서", "PDF순위", "구분", "전략"], sort=False).agg(
    신호줄=("신호일", "size"), 첫신호일=("신호일", "min"), 종목수=("종목코드", "nunique")).to_string())

줄 수: 40,597 · 열 수: 17
열: ['시장', 'TOP3순서', 'PDF순위', '구분', '전략', 'H(고를때)', '신호일', '종목코드', '매수일', '매수가_수정시가', '매수상태', '갭취소', '매수순서값', '20일평균거래대금', '신호출처', '실제로샀다_2012부터', '실제로샀다_2015부터']
신호일: 2012-07-02 ~ 2019-12-30
종목코드 6자리: True

                                                신호줄        첫신호일   종목수
시장     TOP3순서 PDF순위 구분  전략                                           
KOSPI  1      1     나   c6_boll_rsi_oversold   5304  2012-07-09   626
       2      2     다-2 김동겸07_로지스틱상위30%        6906  2012-11-16   677
       3      4     가   disparity              9290  2012-07-02   728
KOSDAQ 1      2     다-2 김동겸07_로지스틱상위30%       11752  2012-11-16  1116
       2      3     나   c6_boll_rsi_oversold   6070  2012-07-10  1067
       3      4     가   rsi+stochastic         1275  2012-07-26   592


In [4]:
# 매수 상태: 체결 = 다음 날 시가에 삼 / 취소_갭 = 다음 날 시가가 너무 높게 시작 / 매수불가 = 다음 날 거래정지 등 /
# 미완료 = 2019-12-30 까지 다 팔지 못함 (또는 마지막 날 신호라 다음 날이 자료 밖)
print(df.groupby(["시장", "구분"], sort=False)["매수상태"].value_counts().unstack(fill_value=0).to_string())
print()
y = pd.to_datetime(df["신호일"]).dt.year
print("해마다 신호 줄 수")
print(df.assign(연도=y).pivot_table(index=["시장", "구분"], columns="연도", values="종목코드", aggfunc="size",
                                   fill_value=0, sort=False).to_string())

매수상태           체결  매수불가  미완료  취소_갭
시장     구분                         
KOSPI  나     5299     2    3     0
       다-2   6894     0   10     2
       가     9270     7    8     5
KOSDAQ 나     6052     5   12     1
       다-2  11716     6   29     1
       가     1271     0    2     2

해마다 신호 줄 수
연도          2012  2013  2014  2015  2016  2017  2018  2019
시장     구분                                                 
KOSPI  나     261   617   631   708   819   550   939   779
       다-2   111   749   780  1482   975   736  1372   701
       가     534   912   934  1792  1381   662  1865  1210
KOSDAQ 나     260   597   531   731  1062   765  1192   932
       다-2   191  1355  1104  2175  1948  1289  2399  1291
       가      35    67    91   272   415   100   159   136


## 3. 신호를 만든 방법

| 기간 | 출처 열 값 | 어떻게 |
|---|---|---|
| 2015~2019 | `해마다(그해전까지로고름)` | PDF 와 같은 신호. 해마다 그 해 시작 21거래일 전까지의 자료로 고른 세트·모델 |
| 2012~2014 | `최종모델(2019-11-29까지로고름)` | 2020 에 쓸 것과 같은 세트·모델 (2019-11-29 까지의 자료로 고름) |

- 가 (기존 1등): 코스피 disparity {ma 20, thr 0.9} (해마다·최종 모두 같은 세트) / 코스닥 rsi+stochastic W=3
  (최종 = rsi {14, 20} + stochastic {14, 3, 20, 80})
- 나 (⑥): 고정 정의 (볼린저(20,2) 하단 + RSI(14) ≤ 30) — 고른 값이 없어 두 기간이 같은 규칙
- 다-2 (김동겸 07): 최종 = 2019-11-29 까지 신호 · 2020 첫날 전에 팔린 매매로 학습한 로지스틱 → 2012~2014 후보를 날짜별 상위 30%
- 매수순서값: 가·나 = 1 (같으면 엔진이 20일 평균 거래대금 큰 순서로 삼), 다-2 = 모델 점수 순번 (클수록 먼저)

In [5]:
src = df.groupby(["시장", "구분", "신호출처"], sort=False).agg(신호줄=("신호일", "size"), 처음=("신호일", "min"),
                                                       끝=("신호일", "max"))
print(src.to_string())
early = pd.to_datetime(df["신호일"]).dt.year < C.WF_YEARS[0]
print()
print("2015 전 줄은 모두 최종 모델:", bool(df.loc[early, "신호출처"].eq(BX.SRC_FINAL).all()))
print("2015 부터 줄은 모두 해마다:", bool(df.loc[~early, "신호출처"].eq(BX.SRC_WF).all()))
print("실제로샀다_2015부터 = 2015 전 빈칸:", bool(df.loc[early, "실제로샀다_2015부터"].isna().all()))

                                   신호줄          처음           끝
시장     구분  신호출처                                               
KOSPI  나   최종모델(2019-11-29까지로고름)  1509  2012-07-09  2014-12-30
           해마다(그해전까지로고름)          3795  2015-01-02  2019-12-26
       다-2 최종모델(2019-11-29까지로고름)  1640  2012-11-16  2014-12-30
           해마다(그해전까지로고름)          5266  2015-01-02  2019-12-30
       가   최종모델(2019-11-29까지로고름)  2380  2012-07-02  2014-12-30
           해마다(그해전까지로고름)          6910  2015-01-02  2019-12-30
KOSDAQ 다-2 최종모델(2019-11-29까지로고름)  2650  2012-11-16  2014-12-30
           해마다(그해전까지로고름)          9102  2015-01-02  2019-12-30
       나   최종모델(2019-11-29까지로고름)  1388  2012-07-10  2014-12-24
           해마다(그해전까지로고름)          4682  2015-01-07  2019-12-26
       가   최종모델(2019-11-29까지로고름)   193  2012-07-26  2014-12-22
           해마다(그해전까지로고름)          1082  2015-01-07  2019-12-16

2015 전 줄은 모두 최종 모델: True
2015 부터 줄은 모두 해마다: True
실제로샀다_2015부터 = 2015 전 빈칸: True


## 4. CSV 만 다시 읽어 엔진에 넣기 → PDF 포트폴리오와 같은가

CSV 의 2015~2019 줄만 날짜×종목 표 (값 = 매수순서값) 로 바꿔 엔진 `run_portfolio` (H5 · 비용 0.5% · 1억 · 20자리) 에 넣는다.
`results/team_compare_portfolio_daily.csv` (PDF 그림의 평가금액) 와 날마다 같으면 통과.
명령이 저장할 때 한 번 확인했고 (`top3_buy_signals_check.csv`), 여기서 **처음부터 다시** 계산한다.

In [6]:
from indexbeater import data_io
from indexbeater import engine as E
from indexbeater import team_compare as T
from indexbeater.exits import ExitRule

data = data_io.load_wide()                         # 기본값 end = 2019-12-31 (2020 이후 안 엶)
dates, codes = data["adj_close"].index, data["adj_close"].columns
saved = pd.read_csv(T.OUT_DAILY, encoding="utf-8-sig", parse_dates=["date"])
saved = saved[(saved["H"] == 5) & (saved["비용"] == "0.5%")]
rows = []
for (mk, g), part in df.groupby(["시장", "구분"], sort=False):
    p = part[pd.to_datetime(part["신호일"]).dt.year >= 2015]
    v = p.pivot(index="신호일", columns="종목코드", values="매수순서값")
    v.index = pd.to_datetime(v.index)
    v = v.reindex(index=dates, columns=codes).fillna(0).astype(int)
    daily, _ = E.run_portfolio(v, data, ExitRule(hold=5), market=mk, cost=C.COST_SELECT)
    eq = daily["equity"][daily.index.year >= 2015]
    ref = saved[(saved["시장"] == mk) & (saved["구분"] == g)].set_index("date")["equity"]
    same = len(ref) == len(eq) and bool((ref.index == eq.index).all())
    diff = float(np.abs(eq.to_numpy() / ref.to_numpy() - 1).max()) if same else np.inf
    rows.append({"시장": mk, "구분": g, "전략": part["전략"].iloc[0], "날짜수": len(eq),
                 "5년수익_CSV": eq.iloc[-1] / 1e8 - 1, "5년수익_PDF": ref.iloc[-1] / 1e8 - 1,
                 "가장큰상대차이": diff, "통과": diff <= 1e-9})
chk = pd.DataFrame(rows)
print(chk.to_string(index=False, formatters={"5년수익_CSV": "{:+.1%}".format, "5년수익_PDF": "{:+.1%}".format,
                                             "가장큰상대차이": "{:.1e}".format}))
print()
print("6칸 모두 통과:", bool(chk["통과"].all()))

    시장  구분                   전략  날짜수 5년수익_CSV 5년수익_PDF 가장큰상대차이   통과
 KOSPI   나 c6_boll_rsi_oversold 1227   +38.6%   +38.6% 2.2e-16 True
 KOSPI 다-2      김동겸07_로지스틱상위30% 1227   +10.5%   +10.5% 2.2e-16 True
 KOSPI   가            disparity 1227   -10.9%   -10.9% 2.2e-16 True
KOSDAQ 다-2      김동겸07_로지스틱상위30% 1227   +41.6%   +41.6% 2.2e-16 True
KOSDAQ   나 c6_boll_rsi_oversold 1227    +7.4%    +7.4% 2.2e-16 True
KOSDAQ   가       rsi+stochastic 1227   +56.7%   +56.7% 2.2e-16 True

6칸 모두 통과: True


## 5. 다-2 최종 모델 진단 (2012~2014 점수)

`학습 안 자료` 주의: 2012~2014 후보는 이 모델의 학습 자료에도 들어 있다 (학습 = 2019-11-29 까지 전부).
그래서 AUC·상위 후보 승률이 실제보다 좋게 보일 수 있다. 참고로만 본다.

In [7]:
f7 = pd.read_csv(BX.OUT_FINAL07, encoding="utf-8-sig")
print(f7.to_string(index=False))

    시장  H      학습cut   학습마지막신호일                점수매긴구간  학습표본  검증후보   실제선택비율      AUC   전체후보승률   상위후보승률                         주의
 KOSPI  5 2019-11-29 2019-11-29 2012-11-16~2014-12-30 20253  4680 0.350427 0.568182 0.510370 0.531422 점수매긴 후보가 학습에도 들어감(학습 안 자료)
KOSDAQ  5 2019-11-29 2019-11-29 2012-11-16~2014-12-30 36170  8044 0.329438 0.568213 0.509704 0.538316 점수매긴 후보가 학습에도 들어감(학습 안 자료)


## 6. 팀원이 쓸 때 주의할 것

1. **매도 학습 자료 자르기**: 시험 해가 Y 면, 학습에는 **Y년 첫날 전에 팔린** 매매만 쓴다 (예정 매도일이 Y년으로 넘어가면 뺀다).
   CSV 에는 매도일이 없다 (매도는 각자 만드는 것이라서). 각자 매도 규칙으로 판 날을 보고 잘라야 한다.
2. **2012~2014 신호는 미래를 본 세트·모델이 고른 것**: 2016~2019 를 본 뒤 고른 세트 (가) · 그 기간까지 학습한 모델 (다-2).
   매도 모델 학습에 쓰면 학습 성적이 좋게 나올 수 있다. 시험 해 (2016~2019) 신호는 모두 '해마다' 출처라 그 해 전 자료로 고른 것.
3. **TOP3 를 2015~2019 결과로 골랐다**: 팀원 시험 해 (2016~2019) 와 겹친다 → 시험 성적도 좋게 나올 수 있다.
   2020 이후 (시험 8) 에서만 정직한 성적을 알 수 있다.
4. **'실제로 샀다' 는 우리 매도 규칙 (H5 = 5거래일 뒤 시가 매도) 기준 참고 열**이다. 매도를 바꾸면 자리가 비는 날이 달라져
   실제로 사는 종목도 달라진다 → 각자 엔진에 신호 전부를 넣고 다시 거르면 된다.
5. **가격 (OHLCV) 은 넣지 않았다**: 각자 가진 데이터를 쓴다. 매수가는 수정 시가 (adj_open) 기준이다.
6. 아직 안 정한 것: 매도 최대 보유 기간을 20일로 둘지.

## 7. 정리

**확인한 것**
- TOP3 CSV 테스트 4개 통과 (1절).
- CSV 40,597줄 · 17열, 신호일 2012-07-02 ~ 2019-12-30, 종목코드 6자리 유지 (2절).
- 2015~2019 줄만 엔진에 다시 넣으면 PDF 포트폴리오 (H5·0.5%) 와 6칸 모두 날마다 같음 (가장 큰 상대 차이 약 2e-16 = 컴퓨터 계산 오차 수준) (4절).
- 2015 전 줄은 모두 '최종 모델', 2015 부터 줄은 모두 '해마다' 출처 (3절).
- 다-2 최종 모델은 2019-11-29 까지 신호 · 2020 첫날 전에 팔린 매매로만 학습 (5절·테스트). 2020 이후 자료는 열지 않음.

**확인 못한 것**
- 팀원이 만든 매도 신호와 합쳤을 때의 결과 (아직 안 받음).
- 2012~2014 신호가 '진짜 그때 알 수 있었던' 신호인지 — 최종 모델로 만들었으니 아니다 (6절 2번). 정직한 성적은 시험 8 (2020 이후) 에서만.
- 매도 최대 보유 20일 유지 여부 (회의에서 정할 것).

**이상한 것**
- 다-2 의 실제 선택 비율이 30% 가 아니라 33~35% — 날짜별 후보 수 × 30% 를 올림해서 (후보가 적은 날 1개라도 고름). PDF 와 같은 방식이라 그대로 둠.
- 다-2 신호는 2012-11-16 부터 시작 (07 모델 특징에 250일 모멘텀이 있어 그 전엔 계산 못 함). 가·나 는 2012-07-02 부터.
- 2019 끝 근처 신호는 '미완료' (5일 안에 다 팔지 못함), 마지막 날 신호 4줄은 매수일·매수가가 빈칸 (다음 날이 자료 밖).
- 다-2 최종 모델의 2012~2014 AUC 0.568 은 학습 안 자료라 좋게 보일 수 있는 값.